# Disappearance: which seizure types go away after surgery

Runs on one **unit** (`UNIT`, the first code cell; see `cc_stats/units.py`): `types`, the seizure-type groups, or `codes`, the individual ILAE 2017 codes with at least 10 patients before surgery. The types notebook is the source; the codes one is generated from it by `analysis/unit_notebooks.py` (only `UNIT` and the notes differ), so edit the types notebook and regenerate. Below, "type" means a group of the unit, and the **core** groups are the unit's `pair_groups`: the five core seizure types, or the codes with at least 20 patients (focal awareness codes aside).

For each seizure-type group, among patients who had it before surgery and whose post-op types were recorded:

- **left**: no longer present after surgery
- **stayed**: still present after surgery

The rates and Wilson 95% CIs come from `disappearance_rates.py`, which this notebook runs. It writes `disappearance_rates.csv` to `results/<version>/2_disappearance/<unit>/`.

In [ ]:
# The unit this notebook runs on: 'types' or 'codes' (cc_stats/units.py).
# Set here only; the codes notebook is generated with 'codes' (analysis/unit_notebooks.py).
UNIT = 'codes'

In [ ]:
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

from cc_stats.paths import ANALYSIS_CSV, DATASET, PROJECT_ROOT, results_dir
from cc_stats.plots import rate_plot, save_figure, stacked_bar
from cc_stats.units import unit as get_unit

analysis = pd.read_csv(ANALYSIS_CSV, index_col='subject_id')
unit = get_unit(UNIT, analysis)
# 'type' or 'code', for figure titles and labels
WORD = 'type' if UNIT == 'types' else 'code'
has_cols = [unit.pre(t) for t in unit.groups]
CORE = unit.pair_groups
CORE_NAME = 'core seizure types' if UNIT == 'types' else 'common seizure codes'
FOLDER = f'2_disappearance/{UNIT}'
# Figures grow with the number of rows (drawn for 10 groups, 5 core)
tall, tall_core = max(1, len(unit.groups) / 10), max(1, len(CORE) / 5)
print(f'dataset {DATASET}')

In [ ]:
RESULTS = results_dir(FOLDER)
run = subprocess.run([sys.executable, 'analysis/2_disappearance/disappearance_rates.py', UNIT],
                     capture_output=True, text=True, cwd=PROJECT_ROOT)
if run.returncode != 0:
    raise RuntimeError(f'disappearance_rates.py failed:\n{run.stderr}')
rates = pd.read_csv(RESULTS / 'disappearance_rates.csv')

In [ ]:
cohort = analysis[analysis[has_cols + [unit.post(t) for t in unit.groups]].notna().all(axis=1)]
print(f'{len(cohort)} patients with pre- and post-op seizure types recorded')

## Core groups: left vs stayed

In [ ]:
counts = pd.DataFrame({
    t: cohort.loc[cohort[unit.pre(t)] == 1, unit.post(t)].map({1: 'stayed', 0: 'left'}).value_counts()
    for t in CORE
}).T.reindex(columns=['left', 'stayed'], fill_value=0).fillna(0).astype(int)
# Most often gone first
counts = counts.loc[(counts['left'] / counts.sum(axis=1)).sort_values(ascending=False).index]
counts.columns.name = 'After surgery'

# The script and this notebook count the same patients; stop if they ever disagree.
check = rates.set_index('type').loc[counts.index]
assert (counts.sum(axis=1) == check['n']).all() and (counts['left'] == check['left']).all()
counts

In [ ]:
ax = stacked_bar(counts, highlight='left', cohort_n=len(cohort), fontsize=10, figsize=(8 * max(1, len(CORE) / 5), 5))
ax.set_title(f'{CORE_NAME.capitalize()}: left vs stayed after surgery')
if max(len(t.get_text()) for t in ax.get_xticklabels()) > 15:  # code labels are long
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', rotation_mode='anchor')
save_figure(FOLDER, 'core_types_left_vs_stayed')
plt.show()

## Disappearance rate, every group

In [ ]:
table = rates.assign(**{
    'left / n': rates['left'].astype(str) + ' / ' + rates['n'].astype(str),
    'rate (95% CI)': [f'{r:.0%} ({lo:.0%}-{hi:.0%})' for r, lo, hi in rates[['rate_left', 'lo', 'hi']].to_numpy()],
})[['type', 'left / n', 'rate (95% CI)']]
table.style.hide(axis='index')

In [ ]:
d = rates.sort_values('rate_left').reset_index(drop=True)
ax = rate_plot(d['type'], d['rate_left'], d['lo'], d['hi'], figsize=(8.5, 5 * tall),
               counts=(d['left'], d['n']), header='Disappeared / total')
ax.set_xlabel(f'Patients whose seizure {WORD} disappeared (Wilson 95% CI)')
ax.set_title('Disappearance rate by ' + ('seizure-type group' if UNIT == 'types' else 'seizure code'), loc='left')
save_figure(FOLDER, 'disappearance_rate_all_groups')
plt.show()

The same for the core groups only.

In [ ]:
d = rates.set_index('type').loc[CORE].sort_values('rate_left')
ax = rate_plot(d.index, d['rate_left'], d['lo'], d['hi'], figsize=(8.5, 3.5 * tall_core),
               counts=(d['left'], d['n']), header='Disappeared / total')
ax.set_xlabel(f'Patients whose seizure {WORD} disappeared (Wilson 95% CI)')
ax.set_title(f'Disappearance rate, {CORE_NAME}', loc='left')
plt.tight_layout()
save_figure(FOLDER, 'disappearance_rate_core_types')
plt.show()